# CCOS — Creative Confidence Operating System
## Scoring Engine · Multi-Format Rubric Registry
### AVC Creative OS · Level 1 Prototype · v6 — Phase 3 Complete

---

**Registry status (v3):**
| Format ID | Platform | Dimensions | Pre-gate | Status |
|---|---|---|---|---|
| `social_static` | Generic Feed | 6 | No | ✅ Updated v3 |
| `youtube_longform` | YouTube Long-Form | 5 | Brand Safety | ✅ Phase 3 v2 |
| `youtube_shorts` | YouTube Shorts | 5 | Platform Compliance | ✅ Phase 3 v1 |
| `meta_feed_static` | Meta Feed (FB + IG) | 6 | Ad Auction Compliance | ✅ Phase 3 v1 |
| `meta_reels` | Meta Reels | 6 | Ad Auction Compliance | ✅ Phase 3 v1 |
| `instagram_reels` | Instagram Reels | 6 | IG Recommendation Eligibility | ✅ Phase 3 v1 |
| `instagram_explore` | Instagram Explore | 5 | IG Recommendation Eligibility | ✅ Phase 3 v1 |
| `instagram_stories` | Instagram Stories | 4 | Stories Basic Compliance | ✅ Phase 3 v1 |
| `linkedin_organic` | LinkedIn Organic Feed | 5 | LinkedIn Organic Eligibility | ✅ Phase 3 v1 |
| `linkedin_sponsored` | LinkedIn Sponsored Update | 5 | Sponsored Update Compliance | ✅ Phase 3 v1 |
| `tiktok_organic` | TikTok Organic (FYP) | 6 | TikTok FYP Eligibility | ✅ Phase 3 v1 |
| `tiktok_ads` | TikTok Ads (FYP + Search) | 5 | TikTok Ads Compliance | ✅ Phase 3 v1 |

**Thresholds (MVP — calibrate with real rejected assets in Alpha):**
- ≥ 70: Launch-ready → proceed to activation
- 55–69: Needs revision → return with critique
- < 55: Do-not-run → rebuild

> ⚠️ `consensus_warning` fires when persona verdict variance is high. The smoothed score must never obscure this signal. Surface it to agency teams.

In [ ]:
# Install dependencies
!pip install anthropic -q

In [ ]:
import anthropic
import json
import base64
from dataclasses import dataclass, field
from typing import Optional, Callable
from pathlib import Path

# Initialise client — API key handled by Colab secrets or env var
client = anthropic.Anthropic()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RUBRIC REGISTRY  (config-driven — add new formats by adding a new entry only)
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class RubricDimension:
    name: str
    weight: float          # must sum to 1.0 across all dimensions in a rubric
    description: str
    scoring_guide: str     # what high / mid / low looks like for the LLM
    calibration_note: str  # benchmark reference from Phase 3 research

@dataclass
class FormatRubric:
    format_id: str
    format_label: str
    dimensions: list[RubricDimension]
    pre_scoring_gate: Optional[dict] = None   # binary compliance check; None = no gate
    platform_context: str = ""                # surface / intent state context for persona prompt
    calibration_notes: str = ""               # overall benchmark summary

# ── SOCIAL STATIC (updated v3 — 6 dimensions) ────────────────────────────────
social_static = FormatRubric(
    format_id="social_static",
    format_label="Social Static (Generic Feed)",
    platform_context="A static image or carousel appearing in a social media feed. The user is scrolling passively. The asset must stop the scroll and motivate a save or share action.",
    dimensions=[
        RubricDimension(
            name="Stop-scroll hierarchy",
            weight=0.20,
            description="Does the visual composition arrest attention within the first 0.5 seconds of scroll?",
            scoring_guide="High: single dominant focal point, high contrast, clear subject. Mid: multiple competing elements, moderate contrast. Low: cluttered, low contrast, no clear hierarchy.",
            calibration_note="First-frame dominance is the primary scroll-arrest mechanism."
        ),
        RubricDimension(
            name="Text-to-image ratio",
            weight=0.15,
            description="Is text usage within platform norms? Does it enhance rather than obscure the visual?",
            scoring_guide="High: <20% text overlay, message clear from visual alone. Mid: 20–30% text, readable. Low: >30% text, cluttered, risks automated suppression.",
            calibration_note="Meta policy threshold: <20% text area. Vision models classify text density at ingestion."
        ),
        RubricDimension(
            name="Native feel",
            weight=0.20,
            description="Does the asset look like organic content rather than an obvious ad? Does it match the platform aesthetic?",
            scoring_guide="High: indistinguishable from organic, UGC aesthetic, native aspect ratio. Mid: polished but not jarring. Low: obviously commercial, stock imagery, disruptive framing.",
            calibration_note="'Feels Like Home' constraint in Meta's candidate selection pipeline penalises disruptive commercial formats."
        ),
        RubricDimension(
            name="Caption + CTA alignment",
            weight=0.10,
            description="Does the caption extend the visual message without repeating it? Is the CTA appropriate to the intent signal (Hot/Warm/Cold)?",
            scoring_guide="High: caption adds context, CTA matches intent state. Mid: caption redundant but not harmful. Low: caption contradicts visual, CTA mismatched to intent.",
            calibration_note="Caption/visual mismatch is a primary aversion trigger — feeds into 'Not Interested' signal."
        ),
        RubricDimension(
            name="Aversion risk",  # NEW v3
            weight=0.20,
            description="Does the asset contain properties likely to trigger 'Not Interested' or 'See fewer posts' actions? Aversion probability is a multiplicative dampener — high aversion suppresses distribution regardless of engagement.",
            scoring_guide="High (low risk): no clickbait, no caption/visual mismatch, density appropriate to feed, no misleading preview. Mid: minor mismatch or slightly dense. Low (high risk): clickbait structure, misleading preview, excessive text, jarring commercial framing.",
            calibration_note="Target aversion rate: <5%. >10% = suppression risk. Must calibrate with real rejected assets."
        ),
        RubricDimension(
            name="Active engagement spurring",  # NEW v3
            weight=0.15,
            description="Is the asset designed to motivate a save or share action specifically? Saves and shares carry significantly higher algorithmic weight than passive likes or watch loops.",
            scoring_guide="High: utility or inspirational value that motivates saving, shareable concept or format, share-prompt implicit in content structure. Mid: likeable but not save/share-worthy. Low: no discernible reason to save or share.",
            calibration_note="Target save rate: >2%, share rate: >1% (top-tier signals on Meta surfaces)."
        ),
    ],
    calibration_notes="Updated v3: +2 dimensions (Aversion Risk, Active Engagement Spurring) grounded in Meta Phase 3 research. Aversion rate target <5%. Save/share rate >2%/>1%."
)

# ── YOUTUBE LONG-FORM (v2 — unchanged) ────────────────────────────────────────
youtube_longform = FormatRubric(
    format_id="youtube_longform",
    format_label="YouTube Long-Form (>3 min, including in-stream ads)",
    pre_scoring_gate={
        "name": "Brand Safety & Advertiser Compliance",
        "description": "Binary check. Fails on: borderline content, muted audio in in-stream ads, metadata not matching spoken content, non-compliant landing page.",
        "fail_action": "Score = 0. Return for revision before dimension scoring runs."
    },
    platform_context="A long-form video on YouTube. The user arrives via Search, Browse, or Suggested. They have active intent (Search) or passive interest (Browse/Suggested). The thumbnail and first 30 seconds determine whether they stay.",
    dimensions=[
        RubricDimension(
            name="Thumbnail strength",
            weight=0.15,
            description="Does the thumbnail generate a competitive CTR without misleading (CVD-detectable clickbait)?",
            scoring_guide="High: visually distinct, face/emotion or clear subject, title reinforces not duplicates thumbnail. Mid: functional but generic. Low: misleading, cluttered, stock imagery.",
            calibration_note="CVD system detects clickbait post-click. Thumbnails that inflate CTR but disappoint on watch time are penalised."
        ),
        RubricDimension(
            name="Opening hook retention (0–30s)",
            weight=0.20,
            description="Does the video retain viewers through the first 30 seconds? Is the keyword/topic spoken within the first 30s?",
            scoring_guide="High: 30-Second Hook Rate target ≥70–80%, keyword spoken early, pattern interrupt within 30s. Mid: clear but slow open. Low: long intro, keyword buried, no hook.",
            calibration_note="30-Second Hook Rate ≥70–80% target. YouTube weights this heavily as an early retention signal."
        ),
        RubricDimension(
            name="Watch time architecture",
            weight=0.25,
            description="Is the video structured to maximise relative watch time? Are pattern interrupts present every 20–40 seconds?",
            scoring_guide="High: MBD z-score positive, pattern interrupt cadence maintained, no dead sections. Mid: mostly engaging with some dead sections. Low: monotonous, no interrupts, viewer drop-off likely.",
            calibration_note="Relative watch time ≥85th cohort percentile. Pattern interrupt cadence: every 20–40 seconds."
        ),
        RubricDimension(
            name="Contextual surface alignment",
            weight=0.25,
            description="Does the video fit the surface it will be delivered on (Search vs Browse vs Suggested)? Is it Trans-Topics capable?",
            scoring_guide="High: topic semantics match likely surface query; LTE-compatible; Trans-Topics path plausible. Mid: fits one surface well. Low: only discoverable via direct search, no cross-surface potential.",
            calibration_note="Trans-Topics discovery is a significant reach multiplier for long-form content."
        ),
        RubricDimension(
            name="Intent-calibrated CTA & messaging",
            weight=0.15,
            description="Is the CTA appropriate to the viewer's intent state on this surface? Does the ad unit (if present) achieve ≥30% view rate?",
            scoring_guide="High: CTA matches Hot/Warm/Cold signal, ad view rate ≥30%, Quality Score ≥7. Mid: CTA present but generic. Low: CTA mismatched or absent.",
            calibration_note="Ad skippable view rate ≥30%. Quality Score ≥7/10. Quality Score acts as CPV divisor — higher score = lower effective CPC."
        ),
    ],
    calibration_notes="Phase 3 v2. 30-Second Hook Rate ≥70–80%. Relative watch time ≥85th cohort percentile. Ad view rate ≥30%. Pattern interrupt every 20–40s."
)

# ── YOUTUBE SHORTS (v1 — unchanged) ───────────────────────────────────────────
youtube_shorts = FormatRubric(
    format_id="youtube_shorts",
    format_label="YouTube Shorts (≤60s / up to 3min with monetisation)",
    pre_scoring_gate={
        "name": "Platform Compliance",
        "description": "Binary check. Fails on: external platform watermarks (TikTok logo etc.), duplicate/reused content, metadata spam, Content ID claims (1–3 min range), wrong aspect ratio.",
        "fail_action": "Score = 0. Silent suppression risk. Return for revision before dimension scoring runs."
    },
    platform_context="A short-form vertical video in the YouTube Shorts feed. The user is in a passive scroll session. Context is sequential — the last 5–10 swipes influence what comes next. The asset has 0–2 seconds to establish its hook before the swipe.",
    dimensions=[
        RubricDimension(
            name="Hook velocity (0–2s / VVSA)",
            weight=0.30,
            description="Does the visual establish context and tension within 0–2 seconds? VVSA (Very Vertical Short Attention) — does it demand the viewer stay?",
            scoring_guide="High: VVSA ≥70% (top-tier), hook established by 1.5s, visual tension or question created. Mid: VVSA 60–70%, hook delayed but present. Low: VVSA <50% (distribution halted threshold), slow open.",
            calibration_note="VVSA ≥70% = top-tier. VVSA <50% = distribution halted. This is the single most weighted signal in the Shorts ranking model."
        ),
        RubricDimension(
            name="Completion rate architecture",
            weight=0.25,
            description="Is the asset structured to achieve ≥80% completion? Is optimal length 15–35 seconds? Is loop engineering present?",
            scoring_guide="High: completion ≥80% likely, length 15–35s, loop designed. Mid: likely 60–80% completion. Low: likely <60%, too long, no loop incentive.",
            calibration_note="Completion rate ≥80% target. Replay rate >15% = loop engineering present. Optimal length: 15–35 seconds."
        ),
        RubricDimension(
            name="Platform nativeness & compliance",
            weight=0.20,
            description="Does the asset look and feel native to YouTube Shorts? Is it free of suppression triggers?",
            scoring_guide="High: 9:16 native, UGC aesthetic, no watermarks, no Content ID risk. Mid: native format but slightly polished. Low: external watermark present, wrong ratio, or reused content.",
            calibration_note="Any external watermark = suppression. Content ID claims in 1–3 min range are a known suppression trigger."
        ),
        RubricDimension(
            name="Contextual & sequential alignment",
            weight=0.15,
            description="Does the asset fit the passive session context? Would it feel coherent after the last 5–10 swipes a user in this audience is likely to have taken?",
            scoring_guide="High: topic fits likely session context, not jarring, topical coherence. Mid: neutral — fits some sessions. Low: jarring, likely to generate aversion swipe.",
            calibration_note="Sequential context fit is a Shorts-specific ranking dimension — the algorithm weights the last N swipes."
        ),
        RubricDimension(
            name="Intent-calibrated CTA",
            weight=0.10,
            description="Is the CTA appropriate to passive intent? Is the related link editor (the only native CTA in Shorts) being used effectively?",
            scoring_guide="High: related link used, CTA matches Cold/Warm intent, no hard sell. Mid: CTA present, slightly mismatched. Low: aggressive CTA, no related link, mismatched intent.",
            calibration_note="Related link editor is the only native CTA mechanism in Shorts. External CTAs feel disruptive and generate aversion."
        ),
    ],
    calibration_notes="Phase 3 v1. VVSA ≥70% (top-tier), <50% = suppressed. Completion ≥80%. Length 15–35s. Replay rate >15%."
)

# ── META FEED STATIC (v1 — NEW) ────────────────────────────────────────────────
meta_feed_static = FormatRubric(
    format_id="meta_feed_static",
    format_label="Meta Feed Static (Facebook Feed + Instagram Feed)",
    pre_scoring_gate={
        "name": "Ad Auction Compliance",
        "description": "Binary check. Fails on: policy-violating content, misleading claims, prohibited product categories, landing page mismatch with ad creative.",
        "fail_action": "Score = 0. Asset blocked from auction entry. Return for revision."
    },
    platform_context="A static image or carousel in the Facebook or Instagram feed. Meta's vision models classify the creative at ingestion — before the auction. Demographic markers, visual stereotypes, and text density are read automatically. The asset competes in a real-time auction where Total Value = (Bid × Pacing Multiplier) + EAR + Ad Quality.",
    dimensions=[
        RubricDimension(
            name="Visual classification safety",
            weight=0.20,
            description="Does the creative avoid stereotypic demographic markers that could cause Meta's vision models to narrow delivery to an unintended demographic? This protects against delivery funnel skew and VRS pacing intervention.",
            scoring_guide="High: demographically neutral or intentionally diverse imagery; no gender/race-coded role stereotypes; visual elements do not predict a narrow target sub-group. Mid: mildly stereotypic but not likely to trigger significant skew. Low: strong stereotypic markers (e.g. gender-coded professional roles, racially coded contexts) that will trigger automated delivery skew.",
            calibration_note="Meta's computer vision classifies demographic markers at ingestion, pre-auction. Research confirms 98% transparent overlay still triggers classification. Must calibrate with real audience delivery data."
        ),
        RubricDimension(
            name="Stop-scroll hierarchy",
            weight=0.20,
            description="Does the visual composition arrest scroll attention within the first 0.5 seconds?",
            scoring_guide="High: single dominant focal point, high contrast, clear subject hierarchy. Mid: multiple competing elements. Low: cluttered, no clear hierarchy, low contrast.",
            calibration_note="Ad Quality score (AQ) in the Total Value equation includes visual clarity signals. High AQ improves auction competitiveness beyond the bid."
        ),
        RubricDimension(
            name="Aversion risk",
            weight=0.20,
            description="Does the asset contain properties likely to generate 'Not Interested' or 'Hide ad' actions? Aversion probability is a multiplicative dampener in Meta's User Value model — high aversion suppresses distribution regardless of engagement scores.",
            scoring_guide="High (low risk): no clickbait, no caption/visual mismatch, density appropriate to feed, no misleading preview, honest framing. Mid: minor mismatch. Low (high risk): clickbait structure, misleading preview, excessive text (>20% overlay), jarring commercial framing.",
            calibration_note="UV(x) = P(like)^w_like × (1 - P(aversion))^w_aversion. Aversion is multiplicative — cannot be offset by high engagement. Target aversion rate: <5%."
        ),
        RubricDimension(
            name="Active engagement spurring",
            weight=0.15,
            description="Is the asset designed to motivate a save or share action? Saves and shares carry significantly higher algorithmic weight than passive likes in Meta's ranking model.",
            scoring_guide="High: utility or inspirational value motivates saving; shareability of concept; share-prompt implicit in content structure. Mid: likeable but not save/share-worthy. Low: no discernible save/share motivation.",
            calibration_note="Save rate >2%, share rate >1% = top-tier signals on Meta. These carry higher weight in EAR prediction than passive likes."
        ),
        RubricDimension(
            name="Native feed integration",
            weight=0.15,
            description="Does the creative blend with the organic feed aesthetic? Meta's candidate selection enforces a 'Feels Like Home' constraint — assets that look like ads are deprioritised.",
            scoring_guide="High: indistinguishable from organic content, UGC aesthetic or editorial framing, native aspect ratio. Mid: polished but not jarring. Low: obviously commercial, stock imagery, disruptive ad framing.",
            calibration_note="'Feels Like Home' merge order: Home Feed seeds >> Recommendation seeds >> Fallback. Native-looking ads earn higher recommendation surface placement."
        ),
        RubricDimension(
            name="Intent-calibrated CTA",
            weight=0.10,
            description="Is the CTA appropriate to the audience's intent signal (Hot/Warm/Cold)? Does it maximise EAR for the target action?",
            scoring_guide="High: CTA matches intent state, action-specific language, EAR-maximising. Mid: generic CTA present. Low: CTA mismatched to intent state, aggressive CTA on Cold audience.",
            calibration_note="EAR (Estimated Action Rate) is the primary variable in Total Value. CTA-intent alignment directly predicts EAR uplift."
        ),
    ],
    calibration_notes="Phase 3 v1. Aversion rate target <5%. Save rate >2%, share rate >1% (top-tier). Text overlay <20%. Classification safety requires calibration with real delivery data."
)

# ── META REELS (v1 — NEW) ────────────────────────────────────────────────────
meta_reels = FormatRubric(
    format_id="meta_reels",
    format_label="Meta Reels (Facebook Reels + Instagram Reels)",
    pre_scoring_gate={
        "name": "Ad Auction Compliance",
        "description": "Binary check — same as Meta Feed Static. Fails on: policy violations, misleading claims, prohibited categories, landing page mismatch.",
        "fail_action": "Score = 0. Asset blocked from auction. Return for revision."
    },
    platform_context="A short-form video in the Facebook or Instagram Reels feed. Meta's Reels ranking model is distinct from the feed: it weights predicted full-completion probability as the baseline signal, and audio page navigation (users creating content with this audio) as a high-value proxy. The UTIS model calibrates ranking against long-term user interest, penalising short-term engagement that does not achieve topical alignment.",
    dimensions=[
        RubricDimension(
            name="Hook velocity (0–1.5s)",
            weight=0.25,
            description="Does the visual AND auditory hook establish context and tension within 1.5 seconds? Meta Reels defaults to audio-on — the audio hook is as important as the visual.",
            scoring_guide="High: visual + audio hook within 1.5s, tension or question established, early completion signal strong. Mid: hook present but delayed (1.5–3s). Low: slow open, no audio hook, context unclear until after 3s.",
            calibration_note="Meta Reels ranking model weights predicted full-completion probability as its baseline signal. Early hook velocity is the primary predictor of completion."
        ),
        RubricDimension(
            name="Audio-first design",
            weight=0.20,
            description="Does the audio carry the narrative independently of the visual? Is the audio designed to motivate audio page navigation (users creating content with this audio track)?",
            scoring_guide="High: audio narrative complete without visuals, inspiring or trending audio, audio page navigation motivation present. Mid: audio supports visual but not independent. Low: audio is background only, not designed for creator re-use.",
            calibration_note="Audio page navigation probability is treated by Meta as a high-value signal — a proxy for creative inspiration. Ranked above passive watch loops in the UTIS-calibrated model."
        ),
        RubricDimension(
            name="Completion pull",
            weight=0.20,
            description="Does the asset structure create a reason to watch to the end? A strong hook is not enough — the viewer must have a reason to stay past the midpoint. Payoff must be positioned at or near the final 3 seconds.",
            scoring_guide="High: unresolved tension or open loop present after 3s, payoff clearly positioned at end, no resolution leak mid-asset. Mid: engaging throughout but no strong end payoff. Low: resolution happens mid-asset, no incentive to complete.",
            calibration_note="Completion probability is the baseline ranking signal. Completion pull is the structural mechanism that achieves it. NEW dimension — requires calibration in Alpha session."
        ),
        RubricDimension(
            name="Active engagement spurring",
            weight=0.15,
            description="Is the asset designed to motivate a save or share? Does it leverage inspiring or trending audio to encourage creator re-use?",
            scoring_guide="High: inspiring or utility-driven, shareability of concept, trending audio leveraged for re-use potential, save motivation present. Mid: likeable but not save/share-worthy. Low: no engagement spur beyond passive watch.",
            calibration_note="Shares, saves, and audio page navigations carry highest weight in Meta Reels ranking. Direct shares and saves significantly outweigh passive watch loops."
        ),
        RubricDimension(
            name="Native surface integration",
            weight=0.10,
            description="Does the creative match the UGC aesthetic of Reels? Is it 9:16 native? Does it avoid external platform watermarks?",
            scoring_guide="High: UGC aesthetic, 9:16 native, no watermarks, 'Feels Like Home' compliance. Mid: native format but visibly produced. Low: external watermarks, wrong ratio, obviously commercial framing.",
            calibration_note="'Feels Like Home' constraint applies on Reels surfaces. External watermarks from competing platforms (TikTok logo etc.) trigger suppression."
        ),
        RubricDimension(
            name="Intent-calibrated CTA",
            weight=0.10,
            description="Is the CTA positioned post-engagement signal? Is it appropriate to the passive-to-active intent arc of a Reels viewer?",
            scoring_guide="High: CTA placed after engagement hook established (not pre-roll), matches Cold/Warm intent arc, no hard sell. Mid: CTA present but early or slightly aggressive. Low: CTA pre-engagement, mismatched to passive intent state.",
            calibration_note="Reels audiences are in a passive exploration state. Hard CTAs pre-engagement generate aversion. CTA should emerge after the hook has been established."
        ),
    ],
    calibration_notes="Phase 3 v1. Completion rate ≥70% target. Audio-on viewing rate ≥80% (Reels default). Save rate >1.5%, share rate >1% top-tier. Hook window: first 1.5s must establish context and tension."
)

# ─────────────────────────────────────────────────────────────────────────────
# RUBRIC REGISTRY — add new formats here only
# ─────────────────────────────────────────────────────────────────────────────

# ── INSTAGRAM REELS (v1) ──────────────────────────────────────────────────────
instagram_reels = FormatRubric(
    format_id="instagram_reels",
    format_label="Instagram Reels (organic + paid)",
    pre_scoring_gate={
        "name": "Instagram Recommendation Eligibility (4-layer)",
        "description": (
            "1. Semantic compliance: no policy violations, violence, regulated substances.\n"
            "2. Technical: 9:16 aspect ratio, no letterboxing, audio present, high-res.\n"
            "3. Originality: no external watermarks, passes duplicate hash check.\n"
            "4. Account health: not under 30-day aggregator suppression (account-level)."
        ),
        "fail_action": "Score = 0. Blocked from recommendation eligibility."
    },
    platform_context=(
        "Instagram Reels. Passive scroll. Distribution governed by the Audition System: "
        "asset first served to a Stage 1 cohort of NON-FOLLOWERS. High engagement velocity "
        "expands to Stage 2 then broad streams. Stage 1 bounce = suppression, no recovery. "
        "No bid override for organic suppression. "
        "Unconnected signal weights: DM Sends w=0.90, Saves w=0.85, Likes w=0.30."
    ),
    dimensions=[
        RubricDimension(
            name="Audition hook velocity (0-3s)",
            weight=0.30,
            description=(
                "Does the asset engineer a strong Stage 1 audition signal within 3 seconds? "
                "The Audition System gates entire distribution trajectory on the first cohort. "
                "No bid recovery for organic suppression. Must establish mute-accessible "
                "context, visual tension, and structural pull by 3s."
            ),
            scoring_guide=(
                "High: context by 1s, tension by 2s, no resolution before 3s, mute-accessible. "
                "Mid: hook present but delayed 2-3s. "
                "Low: slow open, context unclear at 3s, likely Stage 1 bounce."
            ),
            calibration_note=(
                "Hook bounce rate target: <30% at 3s. Completion >=70%. Rewatch >15%. "
                "Calibrate against organic (not paid) performance data."
            )
        ),
        RubricDimension(
            name="DM send potential",
            weight=0.20,
            description=(
                "Is the asset designed to be privately shared via DM? "
                "DM Sends are the dominant unconnected reach signal (w=0.90), "
                "outranking Saves (w=0.85). Must be designed for explicitly."
            ),
            scoring_guide=(
                "High: relatable, emotional, or utility concept — user thinks "
                "'I need to send this to X'. Mid: likeable but not send-worthy. "
                "Low: polished but self-contained, no forwarding motivation."
            ),
            calibration_note=(
                "DM send rate target: >0.5% of reach. "
                "Sends outrank Saves (w=0.90 vs w=0.85). Primary design objective."
            )
        ),
        RubricDimension(
            name="Completion pull and rewatch architecture",
            weight=0.20,
            description=(
                "Does the asset maximise watch time completion and rewatch loops? "
                "Watch time is the most powerful signal (w=0.98). "
                "Loop engineering drives rewatch: end connects back to beginning."
            ),
            scoring_guide=(
                "High: unresolved tension after 3s, payoff at final 3s, loop engineering, "
                "completion >=70% likely, rewatch >15% likely. "
                "Mid: engaging, completion likely, no loop. "
                "Low: resolution mid-asset, no incentive to complete."
            ),
            calibration_note=(
                "Completion target: >=70%. Rewatch: >15%. "
                "Optimal length: 7-15s (highest completion) to 30s (sustained reach)."
            )
        ),
        RubricDimension(
            name="Save and instructional value",
            weight=0.15,
            description=(
                "Does the asset contain educational, reference, or utility value? "
                "Saves weighted w=0.85. Especially high-value for how-to, list, "
                "or educational formats."
            ),
            scoring_guide=(
                "High: clear instructional value, user will want to return, carousel or list format. "
                "Mid: interesting but not reference-worthy. Low: entertainment-only."
            ),
            calibration_note="Save rate target: >1.5% of reach."
        ),
        RubricDimension(
            name="Originality and transformativeness",
            weight=0.10,
            description=(
                "Beyond the watermark check: does the asset meet Instagram's transformativeness "
                "threshold? Voiceovers, original graphics, or native remix required. "
                "Edited reposts fail originality engine even without a watermark."
            ),
            scoring_guide=(
                "High: fully original or clearly transformative. "
                "Mid: adapted with some transformation. "
                "Low: repurposed without transformation, high duplicate hash risk."
            ),
            calibration_note=(
                "Requires: voiceover, original graphics, or native remix. "
                "Accounts over aggregation threshold face 30-day catalog suppression."
            )
        ),
        RubricDimension(
            name="Intent-calibrated CTA",
            weight=0.05,
            description=(
                "CTA appropriate to passive unconnected intent state. "
                "Lowest weight by design. Hard CTAs pre-engagement generate negative signals."
            ),
            scoring_guide=(
                "High: CTA absent or only after full engagement hook established. "
                "Mid: CTA present but not disruptive. "
                "Low: CTA pre-engagement or mismatched to passive intent."
            ),
            calibration_note="Use meta_reels rubric for paid Instagram Reels (higher CTA weight)."
        ),
    ],
    calibration_notes=(
        "Phase 3 v1. Hook bounce <30% at 3s. Completion >=70%. Rewatch >15%. "
        "DM send >0.5%. Save >1.5%. "
        "Audition System: organic suppression is terminal — no bid recovery."
    )
)

# ── INSTAGRAM EXPLORE (v1) ────────────────────────────────────────────────────
instagram_explore = FormatRubric(
    format_id="instagram_explore",
    format_label="Instagram Explore (static, carousel, short video)",
    pre_scoring_gate={
        "name": "Instagram Recommendation Eligibility",
        "description": (
            "Same 4-layer check as instagram_reels. "
            "Thumbnail/cover image quality is the primary entry filter on the Explore grid."
        ),
        "fail_action": "Score = 0. Blocked from Explore recommendation pool."
    },
    platform_context=(
        "Instagram Explore. Active discovery mode — content from accounts not followed. "
        "Candidate generation: co-occurrence interest graphs route via seed account topic clusters. "
        "Ranking: log-linear models predicting save, share, like probability. "
        "NOT passive — navigational intent. "
        "First impression: static thumbnail on dense grid (~110x110px mobile). "
        "Tap/no-tap decision in under 0.5s."
    ),
    dimensions=[
        RubricDimension(
            name="Interest cluster fit",
            weight=0.30,
            description=(
                "Is the asset classifiable into a clear topical cluster? "
                "Co-occurrence graphs route assets via seed account clusters. "
                "Cannot enter candidate pool without a recognisable cluster match."
            ),
            scoring_guide=(
                "High: single dominant topic, visual and caption semantically aligned, "
                "classifiable into a specific niche, hashtags reinforce cluster signal. "
                "Mid: topic identifiable but broad. Low: generic or topically ambiguous."
            ),
            calibration_note=(
                "Interest cluster fit is the retrieval prerequisite. "
                "Without it, quality scores are irrelevant. "
                "Niche specificity outperforms broad appeal on Explore."
            )
        ),
        RubricDimension(
            name="Save and instructional value",
            weight=0.25,
            description=(
                "Does the asset contain reference, educational, or utility value? "
                "Save probability is the primary signal in Explore ranking. "
                "Higher save rate than Reels due to active discovery intent."
            ),
            scoring_guide=(
                "High: clear reference/instructional value, information density warrants saving, "
                "list/how-to/educational format, carousel with actionable steps. "
                "Mid: interesting but not save-worthy. Low: entertainment-only."
            ),
            calibration_note=(
                "Save rate target: >2% (higher than Reels at >1.5% "
                "due to active discovery intent state)."
            )
        ),
        RubricDimension(
            name="Thumbnail and cover image quality",
            weight=0.20,
            description=(
                "Does the thumbnail arrest attention on the Explore grid? "
                "~110x110px static thumbnail. Tap/no-tap in under 0.5s. "
                "Primary gatekeeping mechanism on this surface."
            ),
            scoring_guide=(
                "High: single dominant focal point, high contrast, topic clear at small size, "
                "face or clear subject, text readable at grid scale. "
                "Mid: clear but not arresting. Low: cluttered, topic unclear at grid dimensions."
            ),
            calibration_note=(
                "Must communicate at ~110x110px. "
                "Face-forward images outperform abstract compositions on Explore CTR."
            )
        ),
        RubricDimension(
            name="DM send potential",
            weight=0.15,
            description=(
                "Is the asset designed to be privately shared via DM? "
                "DM Sends: w=0.90 on unconnected reach. "
                "Explore users who discover valuable content often forward it privately."
            ),
            scoring_guide=(
                "High: relatable, emotionally resonant, or highly useful — "
                "user would send to a specific person. "
                "Mid: interesting but self-contained. Low: no forwarding motivation."
            ),
            calibration_note="DM send rate target: >0.5% of reach."
        ),
        RubricDimension(
            name="Threaded comment depth potential",
            weight=0.10,
            description=(
                "Does the asset prompt multi-user conversation threads? "
                "Comment depth weighted w=0.75 on unconnected reach. "
                "Emoji comments and engagement-bait explicitly discounted."
            ),
            scoring_guide=(
                "High: genuine conversation trigger, question or contested claim, "
                "topic invites personal experience sharing, threads likely multi-word. "
                "Mid: commentable but likely emoji-heavy. Low: no conversation trigger."
            ),
            calibration_note=(
                "Comment depth w=0.75 on unconnected reach. "
                "Threaded replies weighted highest. Emoji comments discounted."
            )
        ),
    ],
    calibration_notes=(
        "Phase 3 v1. Save >2%. DM send >0.5%. "
        "Interest cluster fit is the retrieval prerequisite. "
        "Thumbnail is the gatekeeping mechanism."
    )
)

# ── INSTAGRAM STORIES (v1) ────────────────────────────────────────────────────
instagram_stories = FormatRubric(
    format_id="instagram_stories",
    format_label="Instagram Stories (organic brand content)",
    pre_scoring_gate={
        "name": "Stories Basic Compliance",
        "description": (
            "Lighter gate — shown to followers, not recommendation pools. "
            "Fails on: policy violations, aspect ratio not 9:16, audio absent (if video), "
            "misinformation flags."
        ),
        "fail_action": "Score = 0. Return for revision."
    },
    platform_context=(
        "Instagram Stories. High-intimacy, relationship-maintenance surface. "
        "Content from followed accounts. Ranking evaluates DM history, story replies, "
        "mutual tags, profile visits to calculate closeness scores and tray order. "
        "Intent state: relationship upkeep — NOT discovery. "
        "Unpolished, native, conversational formats outperform produced content."
    ),
    dimensions=[
        RubricDimension(
            name="Relationship signal integration",
            weight=0.30,
            description=(
                "Does the asset create an opportunity to signal relationship closeness? "
                "Story tray order governed by closeness scores. "
                "Best-performing Stories create a reason for the viewer to respond."
            ),
            scoring_guide=(
                "High: direct reply prompt, viewer motivated to DM or reply, "
                "two-way interaction created. Mid: interesting but passive only. "
                "Low: broadcast-only, no interaction trigger."
            ),
            calibration_note=(
                "Story reply rate target: >2%. DM from Story: >1%. Profile visit: >1%. "
                "These signals move the brand higher in the tray over time."
            )
        ),
        RubricDimension(
            name="Interactive element usage",
            weight=0.25,
            description=(
                "Are native interactive stickers used effectively? "
                "Polls, questions, sliders, quizzes, countdown — "
                "primary engagement mechanism on Stories, not optional."
            ),
            scoring_guide=(
                "High: sticker directly relevant, question/poll genuinely interesting, "
                "low-friction to respond. Mid: sticker present but generic. "
                "Low: no interactive element."
            ),
            calibration_note="Interactive sticker response rate target: >5%."
        ),
        RubricDimension(
            name="Visual intimacy and authenticity",
            weight=0.25,
            description=(
                "Does the asset match the visual register of Stories? "
                "Over-produced content generates forward-taps (negative signal). "
                "UGC-style, behind-the-scenes, conversational formats match "
                "user expectations on this surface."
            ),
            scoring_guide=(
                "High: UGC or behind-the-scenes aesthetic, imperfect and human, "
                "looks like followed friend's content. Mid: slightly produced but not jarring. "
                "Low: clearly a produced ad, forward-tap risk."
            ),
            calibration_note=(
                "Forward-tap is a negative signal. Completion target: >70%. "
                "UGC register consistently outperforms campaign creative."
            )
        ),
        RubricDimension(
            name="CTA naturalness",
            weight=0.20,
            description=(
                "Is the CTA conversational and appropriate to the relationship context? "
                "Link sticker and question-reply are the only native CTA mechanisms. "
                "Hard-sell CTAs generate forward-taps and damage the closeness score."
            ),
            scoring_guide=(
                "High: CTA feels like natural conversation step, link sticker delivers "
                "genuine value, or question-reply deepens relationship. "
                "Mid: CTA present but slightly transactional. "
                "Low: hard sell or CTA incongruent with content."
            ),
            calibration_note=(
                "Link sticker tap rate target: >3%. "
                "Value-delivery CTAs outperform discount/purchase CTAs."
            )
        ),
    ],
    calibration_notes=(
        "Phase 3 v1. Story reply >2%. Sticker response >5%. Completion >70%. "
        "Relationship maintenance intent, not discovery. "
        "Over-produced content penalised; UGC/conversational rewarded."
    )
)


# ── LINKEDIN ORGANIC (v1) ─────────────────────────────────────────────────────
linkedin_organic = FormatRubric(
    format_id="linkedin_organic",
    format_label="LinkedIn Organic Feed Post",
    pre_scoring_gate={
        "name": "LinkedIn Organic Eligibility",
        "description": (
            "1. Profile-content alignment: author profile headline, skills, and job history "
            "must be semantically aligned with post topic. LLaMA 3 dual-encoder generates "
            "cosine similarity between Member Prompt and Item Prompt — misalignment causes "
            "programmatic suppression before any user sees the post. Manual check required.\n"
            "2. Zero external links in post body: outbound links cause 25-68% reach reduction. "
            "Move links to first comment after early velocity is established.\n"
            "3. Hashtag count 0-2: 3+ hashtags treated as spam signal, triggering 29% reach reduction.\n"
            "4. No coordinated activity: engagement pod participation triggers 60-90 day shadow ban."
        ),
        "fail_action": "Score = 0. Resolve structural issues before dimension scoring runs."
    },
    platform_context=(
        "LinkedIn organic feed. Professional context: career development, industry learning, "
        "network maintenance. ~70% of lifetime reach determined in first 60-90 minutes. "
        "Signal weights: Comments (15x), Shares (5x), Saves (3x), Likes (1x). "
        "Comments in first 5 min: 4.2x distribution multiplier. "
        "Author reply within 1 hour: +30-35% lifetime reach. "
        "LongDwell: >=15s text/image (15.6% subsequent engagement), >=28s video. "
        "Hard negatives fired at <2-3s scroll-past. "
        "Document carousels: 21.77% median engagement, 3x higher than video. "
        "Feed-SR sequential model tracks up to 1,000 professional interactions with RoPE encoding."
    ),
    dimensions=[
        RubricDimension(
            name="Semantic sync and profile grounding",
            weight=0.20,
            description=(
                "Does the post vocabulary map directly to the author's proven professional domain? "
                "LLaMA 3 dual-encoder compares Member Prompt (author profile: headline, skills, "
                "job history) against Item Prompt (post text, media type). "
                "Weak cosine similarity = programmatic suppression. "
                "Generic or off-topic content dilutes the author's Feed-SR sequence vector."
            ),
            scoring_guide=(
                "High: vocabulary maps precisely to author's headline and skills; "
                "domain-specific professional terminology; clearly within established topical pillar. "
                "Mid: partially relevant, adjacent topic, some domain terms. "
                "Low: generic content or topic mismatch with author's professional domain; "
                "high programmatic suppression risk."
            ),
            calibration_note=(
                "Profile-content alignment is the retrieval prerequisite on LinkedIn. "
                "Consistent publishing within a defined professional pillar builds "
                "Feed-SR sequence authority over time."
            )
        ),
        RubricDimension(
            name="Attention hook and dwell architecture",
            weight=0.25,
            description=(
                "Does the post structure maximise passive dwell past the LongDwell threshold? "
                "LongDwell: >=15s text/image, >=28s video. "
                "Assets exceeding thresholds achieve 15.6% subsequent engagement vs 1.2% at 0-3s. "
                "First 2 lines must force a 'see more' click on mobile. "
                "Text blocks broken every 1-2 sentences. "
                "Core value payoff in final paragraph to hold viewport attention."
            ),
            scoring_guide=(
                "High: first 2 lines create curiosity or tension demanding 'see more'; "
                "line breaks every 1-2 sentences; value payoff in final paragraph; "
                "structure guarantees >=15s dwell (text) or >=28s (video). "
                "Mid: clear hook but layout fails to sustain attention past 10s. "
                "Low: wall of text, no hook, high hard-negative scroll-past risk."
            ),
            calibration_note=(
                "LongDwell: >=15s text/image, >=28s video. "
                "15.6% vs 1.2% subsequent engagement at LongDwell vs 0-3s dwell. "
                "Document carousels extend passive dwell naturally as users swipe pages."
            )
        ),
        RubricDimension(
            name="Comment catalyst quality",
            weight=0.20,
            description=(
                "Does the post close with a specific professional question designed to generate "
                "detailed comments within the first 5 minutes? "
                "Comments weighted 15x base. Comments in first 5 min: 4.2x distribution multiplier. "
                "Author replies within 1 hour: +30-35% lifetime reach. "
                "Closing question must prompt long-form, threaded professional discussion — "
                "not yes/no or emoji responses, which are down-weighted."
            ),
            scoring_guide=(
                "High: closes with specific niche professional question requiring domain expertise "
                "to answer meaningfully; likely to generate 3+ detailed comments in first 5 min; "
                "avoids yes/no or engagement-bait formats. "
                "Mid: question present but generic, likely low-effort reactions. "
                "Low: no discussion prompt or vague call-to-action."
            ),
            calibration_note=(
                "4.2x distribution multiplier: comments in first 5 min. "
                "Author reply within 1 hour: +30-35% lifetime reach. "
                "High likes with low dwell time = engagement bait signal = suppression trigger. "
                "Comment seeding playbook required post-publish (separate from asset score)."
            )
        ),
        RubricDimension(
            name="Format and distribution hygiene",
            weight=0.20,
            description=(
                "Is the asset in the highest-performing format with all spam signals removed? "
                "Document carousel: 21.77% median engagement, 3x higher than video. "
                "External links in body: 25-68% reach reduction. "
                "Hashtags >=3: 29% reach reduction. "
                "These are hard algorithmic penalties, not guidelines."
            ),
            scoring_guide=(
                "High: multi-page PDF/document carousel; zero external links in body "
                "(link in first comment only); 0-2 highly specific hashtags. "
                "Mid: zero links, standard image format, 2-3 hashtags. "
                "Low: external links in body, 3+ hashtags, or single-image when "
                "document carousel was viable."
            ),
            calibration_note=(
                "Document carousel: 21.77% median engagement, 3x video. "
                "Short-form video <30s: 89% completion vs 31% for >60s. "
                "LinkedIn Live: up to 24x real-time engagement (live only)."
            )
        ),
        RubricDimension(
            name="Intent state and sequence grounding",
            weight=0.15,
            description=(
                "Does the asset target a specific professional workflow or learning journey? "
                "Feed-SR tracks up to 1,000 professional interactions using RoPE temporal encoding. "
                "Consistent topic publishing builds sequence authority. "
                "Moderately weak tie framing (cross-functional appeal bridging disparate "
                "professional communities) maximises diffusion velocity."
            ),
            scoring_guide=(
                "High: highly specialised actionable breakdown of emerging industry trend or workflow; "
                "domain-specific terminology; cross-functional appeal that bridges communities. "
                "Mid: useful generic advice but lacks specific technical takeaways. "
                "Low: broad motivational platitudes with zero workflow value."
            ),
            calibration_note=(
                "Feed-SR RoPE temporal encoding: professional engagement on Monday 08:00 "
                "weighted differently from Saturday 22:00. Publish during peak windows. "
                "Moderately weak ties (~10 mutual connections) optimise diffusion in digital/tech."
            )
        ),
    ],
    calibration_notes=(
        "Phase 3 v1. LongDwell >=15s text/image, >=28s video. "
        "Comment velocity: 3+ comments in first 5 min = 4.2x multiplier. "
        "Author reply within 1 hour: +30-35% lifetime reach. "
        "Document carousel: 21.77% median engagement. "
        "Zero external links in body. Hashtags 0-2 max. "
        "Profile-content alignment is the retrieval prerequisite."
    )
)

# ── LINKEDIN SPONSORED (v1) ───────────────────────────────────────────────────
linkedin_sponsored = FormatRubric(
    format_id="linkedin_sponsored",
    format_label="LinkedIn Sponsored Update (paid)",
    pre_scoring_gate={
        "name": "Sponsored Update Compliance",
        "description": (
            "1. Profile-content alignment: same as organic — author/brand profile must align "
            "with ad topic for CADET to generate strong embedding similarity.\n"
            "2. Zero external links in ad body: same 25-68% reach penalty applies to paid.\n"
            "3. Audience breadth: overtargeted audiences prevent CADET from learning "
            "sequential historical pathways. Minimum 50,000+ estimated reach recommended.\n"
            "4. CTA-objective alignment: CADET splits chargeability by social interactions vs "
            "landing page clicks — CTA must match the campaign optimisation objective."
        ),
        "fail_action": "Score = 0. Resolve structural issues before dimension scoring runs."
    },
    platform_context=(
        "LinkedIn Sponsored Update. Real-time GSP auction governed by CADET "
        "(Context-Conditioned Ads Decoder-Only Transformer). "
        "CADET models post-scoring contextual features including ad position (K decoding heads) "
        "to resolve position bias. CTR_p = f(x, p) across all candidate positions simultaneously. "
        "Chargeability split by multi-task heads: social interactions vs landing page clicks. "
        "Broader audience definitions allow CADET to learn from sequential user histories. "
        "Thought Leader Ads: 1.7x CTR, 1.6x engagement vs standard single-image. "
        "Same LongDwell thresholds: >=15s text/image, >=28s video."
    ),
    dimensions=[
        RubricDimension(
            name="Semantic sync and profile grounding",
            weight=0.20,
            description=(
                "Does the ad align with the sponsoring author/brand's professional domain? "
                "CADET uses combined user and ad feature embeddings. "
                "Author or brand credibility is a trust signal in paid contexts. "
                "Semantic misalignment reduces predicted CTR. "
                "Especially critical for Thought Leader Ads using named individual profiles."
            ),
            scoring_guide=(
                "High: ad topic maps precisely to author/brand's established domain; "
                "professional vocabulary consistent with profile; credibility signals evident. "
                "Mid: partially relevant, adjacent topic. "
                "Low: topic mismatch with author/brand profile; credibility gap visible."
            ),
            calibration_note=(
                "Author credibility is a paid trust signal as well as an organic ranking input. "
                "Thought Leader Ads require strong named individual profile-content alignment."
            )
        ),
        RubricDimension(
            name="Attention hook and dwell architecture",
            weight=0.20,
            description=(
                "Does the ad structure hold attention past the LongDwell threshold? "
                "Same thresholds as organic: >=15s text/image, >=28s video. "
                "CADET models dwell as a signal quality indicator. "
                "Front-loaded value structure with 'see more' trigger on mobile."
            ),
            scoring_guide=(
                "High: first 2 lines force 'see more'; line breaks every 1-2 sentences; "
                "value payoff in final paragraph; >=15s dwell likely. "
                "Mid: clear hook but layout fails to sustain attention past 10s. "
                "Low: blocky text, no hook, hard negative scroll-past risk."
            ),
            calibration_note=(
                "LongDwell >=15s text/image, >=28s video. "
                "15.6% vs 1.2% subsequent engagement differential applies equally to paid."
            )
        ),
        RubricDimension(
            name="Thought Leader Ad format selection",
            weight=0.25,
            description=(
                "Is the ad using the highest-performing paid format available? "
                "Thought Leader Ads: 1.7x CTR, 1.6x engagement vs standard single-image. "
                "Document Ads: second-highest, leverages same carousel dwell advantage as organic. "
                "Single-image should only be used when Thought Leader or Document formats "
                "are not viable for the campaign objective."
            ),
            scoring_guide=(
                "High: Thought Leader Ad with named individual author and strong profile-content "
                "alignment; or Document Ad with multi-page carousel. "
                "Mid: single-image with strong creative execution. "
                "Low: standard single-image with no creative differentiation."
            ),
            calibration_note=(
                "Thought Leader Ads: 1.7x CTR, 1.6x engagement vs single-image. "
                "Document Ads: 21.77% median engagement. "
                "Short-form video <30s: 89% completion vs 31% for >60s."
            )
        ),
        RubricDimension(
            name="CADET CTR optimisation",
            weight=0.20,
            description=(
                "Is the ad structured to maximise CADET's position-conditioned CTR prediction? "
                "Broad audience definitions allow CADET to learn from sequential user histories — "
                "narrow targeting degrades model performance. "
                "Native lead-gen forms outperform external links in paid. "
                "Always-on ad group structure preferred over burst campaigns."
            ),
            scoring_guide=(
                "High: broad audience (50,000+ estimated reach); native lead-gen form or "
                "CADET-optimised CTA button; zero external links in body; always-on structure. "
                "Mid: moderate targeting with some external link usage. "
                "Low: narrow audience; external links in body; single-burst campaign."
            ),
            calibration_note=(
                "Minimum estimated reach: 50,000+ for CADET sequential learning. "
                "Always-on ad groups outperform burst campaigns. "
                "Native lead-gen forms prevent the external link reach penalty."
            )
        ),
        RubricDimension(
            name="Intent-calibrated CTA alignment",
            weight=0.15,
            description=(
                "Does the CTA match the campaign optimisation objective and the audience's "
                "intent signal (Hot/Warm/Cold)? "
                "CADET splits chargeability: social interactions vs landing page clicks. "
                "Using a conversion CTA on a Cold audience optimised for engagement "
                "generates signal mismatch and degrades auction performance."
            ),
            scoring_guide=(
                "High: CTA precisely matches campaign objective; aligned to audience intent state; "
                "professionally appropriate language. "
                "Mid: CTA present but slightly mismatched to objective or intent state. "
                "Low: conversion CTA on Cold/awareness campaign; objective-CTA mismatch."
            ),
            calibration_note=(
                "CADET chargeability: Hot intent = conversion/demo CTA; "
                "Warm = content/resource CTA; Cold = engagement/awareness CTA. "
                "Mismatch between CTA type and optimisation objective degrades CTR prediction."
            )
        ),
    ],
    calibration_notes=(
        "Phase 3 v1. Thought Leader Ad: 1.7x CTR vs single-image. "
        "LongDwell >=15s text/image, >=28s video. "
        "Audience: 50,000+ estimated reach for CADET learning. "
        "Zero external links in ad body. Native lead-gen forms preferred. "
        "CTA must match campaign optimisation objective."
    )
)


# ── TIKTOK ORGANIC (v1) ──────────────────────────────────────────────────────
tiktok_organic = FormatRubric(
    format_id="tiktok_organic",
    format_label="TikTok Organic (FYP)",
    pre_scoring_gate={
        "name": "TikTok FYP Eligibility",
        "description": (
            "1. Content safety: no graphic medical, legally restricted, spam, or identical duplicates.\n"
            "2. Format: 9:16 vertical, no horizontal framing, no low resolution.\n"
            "3. Topical niche: asset must sit within 1-2 consistent topic clusters for the creator account. "
            "3+ unrelated topics across account posts = 45% reach reduction — "
            "verify account niche consistency before scoring.\n"
            "4. Originality: no duplicate hash match. Transformative content required for repurposed assets."
        ),
        "fail_action": "Score = 0. Blocked from FYP recommendation pool. Return for revision."
    },
    platform_context=(
        "TikTok For You Page (FYP). Passive discovery. Distribution governed by a 3-phase pipeline: "
        "Recall (2,000-3,000 candidate pool) -> Predict (narrows to hundreds via user interaction vectors) "
        "-> Sort (delivers 8 videos to FYP). "
        "Cold-start: Basic Traffic Pool (~500 seed users). High playtime/completion -> "
        "Larger Traffic Pool -> Popular Traffic Pool. "
        "Failure in Basic Traffic Pool = suppression, no recovery. "
        "Signal weights (platform formula): Playtime and play actions = highest; "
        "Shares ~3x higher than likes; Saves = medium-high; Comments = medium; Likes = medium. "
        "Monolith real-time model updates minute-by-minute via Worker-PS architecture. "
        "Concept Drift handled in real time — algorithm adapts to user interest shifts within a session."
    ),
    dimensions=[
        RubricDimension(
            name="Visual hook dynamics (0-3s)",
            weight=0.25,
            description=(
                "Does the asset prevent an instant swipe-away in the first 3 seconds? "
                "Playtime and play actions carry the highest operational weights in TikTok's "
                "scoring formula. An immediate swipe-away (1-2s) fires a heavy negative weight "
                "that suppresses the content category and creator. "
                "Hook must establish high-impact visual or auditory tension immediately. "
                "Hovering for even a fraction of a second registers as immediate interest."
            ),
            scoring_guide=(
                "High: strong visual or auditory impact within 1s; narrative tension or "
                "curiosity gap established by 2s; no resolution before 3s; "
                "swipe-away rate likely <20% at 3s. "
                "Mid: hook present but delayed (2-3s), some tension established. "
                "Low: slow or generic open; context unclear at 3s; high swipe-away risk."
            ),
            calibration_note=(
                "Hook swipe-away target: <20% at 3s (Basic Traffic Pool survival threshold). "
                "Hovering = positive interest signal; instant swipe = heavy negative weight. "
                "Hook must work with audio off — many FYP sessions begin muted."
            )
        ),
        RubricDimension(
            name="Sustained narrative and value peaks",
            weight=0.20,
            description=(
                "Does the asset structure maintain playtime and completion through deliberate "
                "value peaks at seconds 10, 30, and 50? "
                "Engaging-form length (60-180s) is now algorithmically preferred over brevity "
                "when retention is maintained — the historic 'shorter is better' heuristic "
                "no longer applies. Completion rate is co-primary with playtime in the "
                "scoring formula. Single-niche consistency is required: 3+ unrelated topics "
                "across posts reduces reach by 45%."
            ),
            scoring_guide=(
                "High: value peaks at ~10s, ~30s, ~50s; 60-180s length if content warrants it; "
                "narrative progression that earns each additional 10 seconds; "
                "single niche focus maintained throughout. "
                "Mid: engaging but no deliberate value peaks; content holds attention "
                "but relies on hook alone. "
                "Low: front-loaded value only; drops off after first 10s; "
                "topic drift mid-video."
            ),
            calibration_note=(
                "Engaging-form sweet spot: 60-180s (current algorithm preference). "
                "Historical negative correlation with length (beta=-0.25) is no longer dominant. "
                "Value density and sustained attention now outweigh brevity. "
                "Completion rate is a primary traffic pool advancement gate."
            )
        ),
        RubricDimension(
            name="Acoustic-visual sync",
            weight=0.15,
            description=(
                "Are captions and audio perfectly synchronised with the visual narrative? "
                "Synchronized captions/lyrics show up to 40% higher average watch times — "
                "the largest single watch-time premium documented in the registry. "
                "Word-by-word highlighted captions and dynamic text effects maintain attention "
                "and improve accessibility. Trending or brand-aligned audio drives loop behavior "
                "as the algorithm fetches content with matching audio tracks."
            ),
            scoring_guide=(
                "High: word-by-word or lyric-sync captions; audio and visual rhythm aligned; "
                "trending or original audio with strong beat structure; "
                "dynamic text effects used purposefully. "
                "Mid: captions present but static; audio functional but not sync-optimised. "
                "Low: no captions; audio/visual out of sync; generic background music."
            ),
            calibration_note=(
                "Synchronized captions: +40% average watch time. "
                "TikTok-specific mechanic — no equivalent documented multiplier on other platforms. "
                "Algorithm fetches content with matching audio tracks for looping sessions."
            )
        ),
        RubricDimension(
            name="Topical niche consistency",
            weight=0.15,
            description=(
                "Does the asset sit within the creator's established topical niche cluster? "
                "Posting across 3+ unrelated topics reduces reach by 45% — the most severe "
                "topical consistency penalty in the registry. The algorithm builds "
                "creator authority within defined niche embeddings over time. "
                "A single off-niche video degrades the creator's sequence vector."
            ),
            scoring_guide=(
                "High: asset clearly within 1-2 established topic clusters for this creator; "
                "vocabulary and visual language consistent with creator's existing content; "
                "no topic drift from established niche. "
                "Mid: adjacent topic — related but not core niche. "
                "Low: unrelated topic; significantly different from creator's established niche; "
                "45% reach reduction risk."
            ),
            calibration_note=(
                "45% reach reduction for 3+ unrelated topic posts. "
                "Account-level accumulation — topic drift affects all subsequent content, "
                "not just the offending post. "
                "Niche consistency builds embedding authority; topic-switching dilutes it."
            )
        ),
        RubricDimension(
            name="Share and loop engineering",
            weight=0.15,
            description=(
                "Is the asset designed to generate shares and replays? "
                "Share rate is weighted ~3x higher than likes in TikTok's recommendation model. "
                "Loops and re-watches signal exceptional engagement — the algorithm fetches "
                "content with matching audio/hashtag/visual elements when loops are detected. "
                "Saves signal instructional value and keep the asset in circulation over time."
            ),
            scoring_guide=(
                "High: highly relatable, useful, or entertaining concept that prompts "
                "peer-to-peer sharing; seamless loop design (end connects to beginning or "
                "audio loop point); step-by-step or tutorial format motivating saves. "
                "Mid: shareable concept but no explicit loop engineering. "
                "Low: self-contained narrative with no sharing or loop motivation."
            ),
            calibration_note=(
                "Share rate: ~3x weight vs likes. "
                "Loop rate target: >15% (consistent with instagram_reels benchmark). "
                "Saves = instructional value signal; step-by-step and tutorial formats "
                "outperform entertainment-only on save rate."
            )
        ),
        RubricDimension(
            name="Interactive engagement prompt",
            weight=0.10,
            description=(
                "Does the asset include a deliberate prompt for comments, shares, or saves "
                "positioned in the final seconds? "
                "CTA cards in final seconds drive a documented 30% CVR lift. "
                "Direct questions spark comment section debate and threaded replies. "
                "Prompts for peer-to-peer sharing directly target the highest-weighted "
                "active signal in the TikTok model."
            ),
            scoring_guide=(
                "High: CTA card or direct question in final 3s; question is specific enough "
                "to generate genuine replies; share prompt or save prompt embedded naturally. "
                "Mid: call-to-action present but generic or early in the video. "
                "Low: no engagement prompt; passive consumption only."
            ),
            calibration_note=(
                "CTA cards in final seconds: +30% CVR lift. "
                "Direct questions targeting comment debate generate medium-weight "
                "signals that compound over the post's lifetime."
            )
        ),
    ],
    calibration_notes=(
        "Phase 3 v1. Hook swipe-away <20% at 3s. Completion rate = primary traffic pool gate. "
        "Acoustic-visual sync: +40% watch time. Engaging-form 60-180s preferred. "
        "Niche consistency: 3+ unrelated topics = 45% reach reduction. "
        "Share rate: ~3x weight vs likes. Loop rate >15%."
    )
)

# ── TIKTOK ADS (v1) ───────────────────────────────────────────────────────────
tiktok_ads = FormatRubric(
    format_id="tiktok_ads",
    format_label="TikTok Ads (paid — FYP + Search)",
    pre_scoring_gate={
        "name": "TikTok Ads Compliance",
        "description": (
            "All FYP Eligibility checks PLUS paid-specific requirements:\n"
            "1. Format: no horizontal framing, no flipped orientations, no low resolution, "
            "no spelling errors, no URL-formatted titles, no still images <5s.\n"
            "2. Landing page: must be live, complete, and free of inescapable pop-up elements.\n"
            "3. Campaign budget: must meet minimum CPA multiplier requirements — "
            "MAI: budget >= 50x target CPA; AEO mid-funnel: >= 20x CPA; "
            "purchase AEO/VBO: >= 10x CPA. Underfunded campaigns cannot generate enough "
            "data for Monolith to optimise delivery.\n"
            "4. Search Ads (if applicable): keyword list >= 50 terms; "
            "budget-to-bid ratio >= 20:1."
        ),
        "fail_action": "Score = 0. Resolve compliance and budget issues before dimension scoring runs."
    },
    platform_context=(
        "TikTok paid advertising. Two intent states: "
        "(1) Passive FYP — Observe-and-Expand broad targeting. System launches with exact targeting, "
        "observes converters outside specified targeting, automatically expands parameters. "
        "Monolith updates model minute-by-minute via Worker-PS architecture. "
        "(2) Active Search — Deep Retrieval (K^D path beam search) matches queries against "
        "50+ themed keyword groups. Carousel for traffic objectives, Video for conversion. "
        "Full-funnel compounding: brand + performance = +9.9% CVR lift, -10.9% CPA. "
        "Complete funnel: 4.3x more conversions, 36% more efficient CPA, 33% higher CVR "
        "vs single-objective campaigns."
    ),
    dimensions=[
        RubricDimension(
            name="Visual hook dynamics (0-3s)",
            weight=0.25,
            description=(
                "Does the ad prevent an instant swipe-away in the first 3 seconds? "
                "Paid assets compete in the same attention environment as organic — "
                "the Predict phase applies the same early retention evaluation. "
                "Playtime and play actions carry the highest operational weights. "
                "Ad creative quality directly affects auction competitiveness."
            ),
            scoring_guide=(
                "High: strong visual or auditory impact within 1s; tension by 2s; "
                "no resolution before 3s; swipe-away rate likely <20%. "
                "Mid: hook present but delayed. Low: slow open; high skip risk."
            ),
            calibration_note=(
                "Hook swipe-away target: <20% at 3s. "
                "Paid assets are evaluated against the same early retention thresholds as organic. "
                "Ad quality score affects auction clearing price."
            )
        ),
        RubricDimension(
            name="Sustained narrative and value peaks",
            weight=0.20,
            description=(
                "Does the ad structure maintain completion through deliberate value peaks? "
                "Completion rate is tracked for ad quality scoring. "
                "Same engaging-form length preference (60-180s) applies when campaign objective "
                "warrants longer-form content. Entertainment + effectiveness combination "
                "documented as most efficient full-funnel approach."
            ),
            scoring_guide=(
                "High: value peaks at ~10s, ~30s, ~50s; "
                "narrative earns each additional 10 seconds; entertainment and value combined. "
                "Mid: engaging but no deliberate peaks; front-loaded. "
                "Low: drops off after first 10s."
            ),
            calibration_note=(
                "Engaging-form 60-180s preferred for conversion objectives. "
                "Entertainment + effectiveness combination: documented superior conversion rates."
            )
        ),
        RubricDimension(
            name="Acoustic-visual sync",
            weight=0.15,
            description=(
                "Are captions and audio perfectly synchronised? "
                "Same +40% watch time premium applies to paid creative. "
                "Branded sound design or trending audio with word-by-word captions "
                "are the highest-performing audio configurations for paid TikTok."
            ),
            scoring_guide=(
                "High: word-by-word captions synced to audio; dynamic text effects; "
                "trending or brand-aligned audio. "
                "Mid: captions present but static. Low: no captions or audio/visual desync."
            ),
            calibration_note="+40% average watch time for synchronized captions. Applies equally to paid."
        ),
        RubricDimension(
            name="Ad hygiene and format compliance",
            weight=0.20,
            description=(
                "Does the ad meet all paid-specific format requirements that prevent "
                "automatic throttling or suppression? "
                "Paid suppression triggers are more specific than organic: "
                "horizontal flips, low resolution, spelling errors, URL-formatted titles, "
                "and still images <5s all trigger automatic throttle before auction entry. "
                "Landing page quality directly affects ad delivery — broken or "
                "pop-up-heavy pages cause suppression."
            ),
            scoring_guide=(
                "High: 9:16 native; no technical format violations; no spelling errors; "
                "title clearly branded (not URL); still images >=5s if used; "
                "landing page live, complete, and clean. "
                "Mid: minor technical issues not likely to trigger throttle. "
                "Low: format violations present; landing page issues; "
                "spelling errors or URL title."
            ),
            calibration_note=(
                "Paid-specific suppression triggers: horizontal framing, flipped orientation, "
                "low resolution, spelling errors, URL titles, still images <5s, "
                "broken/pop-up landing pages."
            )
        ),
        RubricDimension(
            name="Campaign and audience architecture",
            weight=0.20,
            description=(
                "Is the campaign structured to meet Monolith's data requirements for optimisation? "
                "Underfunded campaigns cannot generate enough data points for the "
                "parameter server to learn and expand delivery effectively. "
                "Observe-and-Expand broad targeting allows the ML model to identify "
                "converters outside static demographic constraints. "
                "Full-funnel strategy (upper + mid + lower) compounds performance significantly."
            ),
            scoring_guide=(
                "High: budget meets CPA multiplier requirements (MAI 50x, AEO 20x, VBO 10x); "
                "broad audience definition for Observe-and-Expand learning; "
                "full-funnel campaign structure; Search Ads with 50+ keywords and 20:1 ratio. "
                "Mid: budget meets minimums but audience slightly narrow. "
                "Low: budget below minimums; narrow targeting preventing ML optimisation; "
                "single-objective campaign only."
            ),
            calibration_note=(
                "CPA multipliers: MAI 50x, AEO 20x, VBO 10x. "
                "Full-funnel: 4.3x more conversions, 36% more efficient CPA, 33% higher CVR. "
                "Brand + performance combined: +9.9% CVR, -10.9% CPA. "
                "Split test: min 2 weeks, 50+ conversions/week per ad group."
            )
        ),
    ],
    calibration_notes=(
        "Phase 3 v1. Hook swipe-away <20% at 3s. Acoustic sync +40% watch time. "
        "CPA multipliers: MAI 50x, AEO 20x, VBO 10x. "
        "Full-funnel: 4.3x more conversions vs single-objective. "
        "Observe-and-Expand: broad targeting required for ML optimisation."
    )
)

RUBRIC_REGISTRY: dict[str, FormatRubric] = {
    "social_static":    social_static,
    "youtube_longform": youtube_longform,
    "youtube_shorts":   youtube_shorts,
    "meta_feed_static": meta_feed_static,
    "meta_reels":       meta_reels,
    "instagram_reels":   instagram_reels,
    "instagram_explore": instagram_explore,
    "instagram_stories": instagram_stories,
    "linkedin_organic":   linkedin_organic,
    "linkedin_sponsored": linkedin_sponsored,
    "tiktok_organic":     tiktok_organic,
    "tiktok_ads":         tiktok_ads,
}

def route_to_rubric(format_type: str) -> FormatRubric:
    """Route an asset to its rubric. Raises ValueError for unregistered formats."""
    if format_type not in RUBRIC_REGISTRY:
        raise ValueError(
            f"Format '{format_type}' not in rubric registry. "
            f"Available: {list(RUBRIC_REGISTRY.keys())}"
        )
    return RUBRIC_REGISTRY[format_type]

print(f"Rubric registry loaded: {list(RUBRIC_REGISTRY.keys())}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PRE-SCORING GATE CHECK
# ─────────────────────────────────────────────────────────────────────────────

def check_pre_scoring_gate(rubric: FormatRubric) -> bool:
    """Print gate requirements and confirm with the user before scoring.
    Returns True if gate passes (or no gate), False if user reports failure."""
    if rubric.pre_scoring_gate is None:
        print(f"ℹ️  No pre-scoring gate for '{rubric.format_id}'. Proceeding to dimension scoring.")
        return True
    
    gate = rubric.pre_scoring_gate
    print(f"\n{'='*60}")
    print(f"PRE-SCORING GATE: {gate['name']}")
    print(f"{'='*60}")
    print(f"\n{gate['description']}")
    print(f"\nFail action: {gate['fail_action']}")
    print(f"\n{'='*60}")
    
    result = input("Does the asset PASS this gate? (yes/no): ").strip().lower()
    if result != "yes":
        print("\n⛔ Gate FAILED. Score = 0. Asset returned for revision.")
        return False
    print("\n✅ Gate PASSED. Proceeding to dimension scoring.")
    return True

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SYNTHETIC PERSONA DEFINITIONS (Bridge Evaluation — placeholder)
# Target state: personas loaded from BigQuery persona_profiles table
# Surface context field added v3 — required for cross-surface evaluation
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class SyntheticPersona:
    name: str
    description: str
    intent_signal: str          # Hot / Warm / Cold
    surface_context: str        # NEW v3: which surface/platform this persona is on
    behavioural_traits: list[str]
    cultural_sensitivity: list[str] = field(default_factory=list)

# Default personas — replace with client-specific 1PD personas in production
DEFAULT_PERSONAS = [
    SyntheticPersona(
        name="High-Intent Converter",
        description="Has already researched the product category. Ready to act if the right offer appears.",
        intent_signal="Hot",
        surface_context="Meta Feed (connected intent state — Home Feed)",
        behavioural_traits=[
            "Scans for price/value signals immediately",
            "Will click if CTA is specific and low-friction",
            "Dismisses vague brand messaging",
            "Responsive to social proof and specificity"
        ]
    ),
    SyntheticPersona(
        name="Passive Discoverer",
        description="Not actively searching. Open to discovery. Can be inspired but not pushed.",
        intent_signal="Cold",
        surface_context="Meta Reels / Instagram Reels (unconnected discovery surface)",
        behavioural_traits=[
            "Scrolls quickly — stops only for strong visual/audio hooks",
            "Saves content for later if it feels useful or inspiring",
            "Hostile to hard sell or aggressive CTAs",
            "Responds to native, UGC-style content",
            "Audio-on by default on Reels surfaces"
        ]
    ),
    SyntheticPersona(
        name="Warm Considerer",
        description="Aware of the category. Has engaged with similar content before. Evaluating options.",
        intent_signal="Warm",
        surface_context="YouTube Browse / Suggested (passive interest state)",
        behavioural_traits=[
            "Will watch 30–60s if the opening earns it",
            "Compares against known alternatives",
            "Responds to clear differentiation and social proof",
            "Likely to share if content is genuinely useful"
        ]
    ),
]

print(f"{len(DEFAULT_PERSONAS)} personas loaded.")
for p in DEFAULT_PERSONAS:
    print(f"  • {p.name} [{p.intent_signal}] — {p.surface_context}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SCORING ENGINE
# ─────────────────────────────────────────────────────────────────────────────

def build_scoring_prompt(
    rubric: FormatRubric,
    persona: SyntheticPersona,
    asset_description: str
) -> str:
    """Build the evaluation prompt for a single persona × rubric."""
    dimensions_text = "\n".join([
        f"### {d.name} (weight: {d.weight:.0%})\n"
        f"What to evaluate: {d.description}\n"
        f"Scoring guide: {d.scoring_guide}\n"
        f"Calibration note: {d.calibration_note}"
        for d in rubric.dimensions
    ])
    
    return f"""You are evaluating a creative asset as the following synthetic persona.

## Your Persona
Name: {persona.name}
Intent signal: {persona.intent_signal}
Surface context: {persona.surface_context}
Behavioural traits:
{''.join(f'- {t}' + chr(10) for t in persona.behavioural_traits)}

## Platform Context
{rubric.platform_context}

## Asset Description
{asset_description}

## Your Task
Score the asset on each dimension below from 0–100. Be specific and critical.
Then compute the weighted composite score (CCS).

{dimensions_text}

## Response Format (JSON only — no preamble)
{{
  "persona": "{persona.name}",
  "dimension_scores": {{
    {', '.join([f'"{d.name}": <score_0_to_100>' for d in rubric.dimensions])}
  }},
  "ccs": <weighted_composite_0_to_100>,
  "verdict": "launch-ready" | "needs-revision" | "do-not-run",
  "critique": "<2–3 sentence critique specific to this persona's perspective>"
}}"""


def score_asset(
    asset_description: str,
    format_type: str,
    personas: list[SyntheticPersona] = DEFAULT_PERSONAS,
    asset_image_path: Optional[str] = None
) -> dict:
    """Score an asset against all personas and return the CCS with consensus analysis."""
    rubric = route_to_rubric(format_type)
    
    # Gate check
    if not check_pre_scoring_gate(rubric):
        return {"ccs": 0, "verdict": "gate-failed", "persona_scores": [], "consensus_warning": False}
    
    persona_results = []
    
    for persona in personas:
        prompt = build_scoring_prompt(rubric, persona, asset_description)
        
        # Build message content
        content = []
        if asset_image_path:
            with open(asset_image_path, "rb") as f:
                img_data = base64.standard_b64encode(f.read()).decode("utf-8")
            ext = Path(asset_image_path).suffix.lower()
            media_type = {'.jpg': 'image/jpeg', '.jpeg': 'image/jpeg',
                          '.png': 'image/png', '.gif': 'image/gif',
                          '.webp': 'image/webp'}.get(ext, 'image/jpeg')
            content.append({"type": "image", "source": {"type": "base64", "media_type": media_type, "data": img_data}})
        content.append({"type": "text", "text": prompt})
        
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1000,
            messages=[{"role": "user", "content": content}]
        )
        
        raw = response.content[0].text.strip()
        # Strip markdown fences if present
        raw = raw.replace("```json", "").replace("```", "").strip()
        result = json.loads(raw)
        persona_results.append(result)
    
    # Compute consensus
    verdicts = [r["verdict"] for r in persona_results]
    scores = [r["ccs"] for r in persona_results]
    mean_ccs = sum(scores) / len(scores)
    
    verdict_categories = set(verdicts)
    consensus_warning = len(verdict_categories) == 3  # All three personas disagree
    
    # Final verdict from mean CCS
    if mean_ccs >= 70:
        final_verdict = "launch-ready"
    elif mean_ccs >= 55:
        final_verdict = "needs-revision"
    else:
        final_verdict = "do-not-run"
    
    return {
        "format": format_type,
        "ccs": round(mean_ccs, 1),
        "verdict": final_verdict,
        "consensus_warning": consensus_warning,
        "persona_scores": persona_results,
        "score_range": [min(scores), max(scores)],
        "calibration_note": "⚠️ Thresholds are MVP placeholders. Calibrate with real rejected assets in Alpha session."
    }

print("Scoring engine ready.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RESULTS DISPLAY
# ─────────────────────────────────────────────────────────────────────────────

def display_results(result: dict):
    """Print a formatted scoring report."""
    print("\n" + "="*60)
    print(f"  CREATIVE CONFIDENCE SCORE (CCS)")
    print("="*60)
    print(f"  Format:   {result['format']}")
    print(f"  Score:    {result['ccs']} / 100")
    print(f"  Verdict:  {result['verdict'].upper()}")
    print(f"  Range:    {result['score_range'][0]} – {result['score_range'][1]} across personas")
    
    if result.get('consensus_warning'):
        print("\n  ⚠️  CONSENSUS WARNING")
        print("  All three personas returned different verdicts.")
        print("  This may indicate cross-surface performance inconsistency.")
        print("  Do not smooth this signal — surface it to the Signal Engineer.")
    
    print("\n" + "-"*60)
    print("  PER-PERSONA BREAKDOWN")
    print("-"*60)
    for p in result['persona_scores']:
        print(f"\n  {p['persona']} — {p['ccs']}/100 [{p['verdict']}]")
        print(f"  {p['critique']}")
        print("  Dimensions:")
        for dim, score in p['dimension_scores'].items():
            bar = '█' * (score // 10) + '░' * (10 - score // 10)
            print(f"    {dim[:35]:<35} {bar} {score}")
    
    print("\n" + "-"*60)
    print(f"  {result['calibration_note']}")
    print("="*60 + "\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN SCORING — edit asset_description and format_type to evaluate your asset
# ─────────────────────────────────────────────────────────────────────────────

asset_description = """
Replace this with your asset description.

For a static ad: describe the visual composition, text overlay, colour palette,
subject matter, brand elements, and the caption/CTA.

For a video ad: describe the hook (0–3s), narrative arc, audio design, pacing,
CTA placement, and ending.
"""

format_type = "meta_feed_static"  # Change to: social_static | youtube_longform |
                                   #   youtube_shorts | meta_feed_static | meta_reels

# Optional: provide a path to a static image asset for multimodal evaluation
# asset_image_path = "/content/your_asset.jpg"
asset_image_path = None

# Run
result = score_asset(
    asset_description=asset_description,
    format_type=format_type,
    asset_image_path=asset_image_path
)
display_results(result)

---
## What Comes Next

### Phase 3 (continuing in parallel)
**Phase 3 is complete.** All five platforms researched and integrated.

Registry: 12 rubrics across YouTube, Meta, Instagram, LinkedIn, and TikTok.

Next: **Phase 4 — Build the FastAPI backend, PostgreSQL schema, and full scoring pipeline.** Each produces a rubric input document → Phase 4 integration → new `FormatRubric` entry in the registry.

### Open Architecture Decisions
- **M-01** — Platform Distribution Risk module: standalone or rubric sub-layer? (HDC + Engineering)
- **M-02** — Should `SyntheticPersona.surface_context` (added v3) be reflected in the BigQuery schema before Persona Engine v1 is built? (Engineering + HDC)
- **M-05** — Does the 0s/3s/10s keyframe strategy have sufficient signal to score Completion Pull? (Engineering)

### Phase 4 (as each platform research returns)
Add a new `FormatRubric` to `RUBRIC_REGISTRY`. No other code changes required.

### Alpha Calibration (Weeks 19–20)
Run 20+ assets through the system — **use real rejected assets from past campaigns, not hypotheticals**. Set thresholds from actual score distributions, not the 70/55 MVP placeholders.